# Create database and JSON ssot from ODM

```python3 webfillallin1.py [path to your]/IM de FORCE```

In [ ]:
import sys
import os
import glob

In [ ]:
# Add toolbox to python library path
sys.path.insert(0, os.path.abspath('../../pythonWork/pythonSource'))
sys.path.insert(1, os.path.abspath('../../pythonWork/pythonSource/IM_db'))
import IM_db.createDB as cdb
import IM_ODM.fillDB as fdb

In [ ]:
odm_source_folder = '/Users/bue/dev/geberit/DEAP/IM'
odm_source_folder = '/Users/bue/dev/fyyccim-im/ModellModell'

In [ ]:
assert os.path.isdir(odm_source_folder), 'invalid ODM source folder {}'.format(odm_source_folder)

In [ ]:
def bla(condition, *args):
    addon = ('0', *args)
    print('Type of args: {}'.format(type(args)))
    print('Addon {}'.format(addon) )
    print(condition.format(*addon))
    
bla("format {} {} {}", 'a', 'b')

In [ ]:
bla("text", 'su', 1)
bla("text")

In [ ]:
def valuepairs2sqlexpr(**colvalues):
    """input: {colname:colvalue,}
       return "(col-name is NULL or col-name = value)" (depending on colvalue) and concatenated for every colname/-value pair
       if value is not of integer type, enclose it with '' """
    sqlstring = lambda val: "'{}'".format(val) if type(val) != int else str(val)
    comp = lambda col, val: "{} is null".format(col) if val is None else "{} = {}".format(col, sqlstring(val))
    retval = " and ".join("({})".format(comp(col, val)) for col, val in colvalues.items())
    return retval

valuepairs2sqlexpr(x=1, y='a')

In [ ]:
def valuepairs2sqlexpr2(**colvalues):
    condition = ''
    for key in colvalues:
        if len(condition) > 0:
            condition = condition + ' and '
        condition = condition + '{} = ?'.format(key)
    arguments = list(colvalues.values())
    return '({})'.format(condition), *arguments

valuepairs2sqlexpr2(x=1, y='a')

## Initialize parameters

In [ ]:
from IM_DB import parameters
parameters.initparam(odm_source_folder)

## Create empty database

In [ ]:
os.makedirs(parameters.dbDirect(), exist_ok=True)

from IM_ODM import fillDB
from IM_db import existsDB
import contextlib

print('Creating database {db} from model {odm}'.format(db=parameters.dbFilePath(), odm=parameters.dbDirect()))

with contextlib.suppress(FileNotFoundError):
    os.remove(parameters.dbFilePath())
    
try:
    fillDB.filldbmain2(odm_source_folder, createnewdb=not existsDB(parameters.dbFilePath()))
except:
    print('Consult logfile {}'.format(parameters.logfilepath()))
    raise
#create_db_result = cdb.createDB(odm_source_folder)
#create_db_result

## Find and validate the database

In [ ]:
dbfiles = glob.glob(odm_source_folder + '/DB/*.db')
assert len(dbfiles) == 1, 'Expecting exactly one database. Found {}'.format(dbfiles)

database_file = os.path.abspath(dbfiles[0])
'Working with database {}'.format(database_file)

In [ ]:
from IM_DB import dbConnect
from IM_JSON import sql2json,JSModel
    
dbConnect.openDB(parameters.dbFilePath(), fks='ON')
jsmodel = JSModel(pmodel=sql2json(pdbname=parameters.dbFilePath()))
# printHTML.setWebDirec(p_webdirec=None)
# listWebdoku.listwebmain(plang=Languagetext.reportLang(),pmodel=jsmodel)
json_file = jsmodel.printmodel(pfilepath=parameters.dbDirect(),pfilename=parameters.odmModelName())
print(json_file)
dbConnect.closeDB()

In [ ]:
import json

with open(json_file, 'r') as source:
    reload = json.load(source)

In [ ]:
for root in reload:
    print('Root {} contains {}'.format(root, len(reload[root])))